# Clean H3 Features - Silver Layer

Ingests CARTO Marketplace H3 data, filters to target states, and creates derived features for ML model.

**Data Source:** CARTO Marketplace - Derived Spatial Features USA H3 Resolution 8

**Target States:** MA, CT, NJ, MD (Northeast region for sales prediction model)

**Output:**
- `{catalog}.{silver_schema}.h3_features_clean` - Clean H3 features with derived columns

**Derived Features:**
- `target_demographic_total`: Sum of ages 20-34 (male + female)
- `total_poi_count`: Sum of all POI categories
- `urbanity_category`: Standardized urbanity categories for K-ring sizing

## Parameters

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when, coalesce, array

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("carto_table", "carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3")
dbutils.widgets.text("state_filter", "MA,CT,NJ,MD")  # Comma-separated state abbreviations

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
carto_table = dbutils.widgets.get("carto_table")
state_filter = dbutils.widgets.get("state_filter")

# Parse state filter into list
state_list = [s.strip() for s in state_filter.split(",") if s.strip()]

assert catalog and bronze_schema and silver_schema, "Missing required parameters"
assert len(state_list) > 0, "At least one state must be specified in state_filter"

# Table names
states_table = f"{catalog}.{bronze_schema}.census_states"
output_table = f"{catalog}.{silver_schema}.h3_features_clean"

print(f"Catalog: {catalog}")
print(f"CARTO source: {carto_table}")
print(f"State filter: {state_list}")
print(f"Output table: {output_table}")

## Validate and Load CARTO Marketplace Data

In [ ]:
# Load CARTO marketplace table
try:
    carto_df = spark.table(carto_table)
    carto_count = carto_df.count()
    print(f"✓ CARTO table found: {carto_table}")
    print(f"  Total H3 cells: {carto_count:,}")
except Exception as e:
    print(f"\n❌ ERROR: CARTO table not found: {carto_table}")
    print(f"\nPlease ensure CARTO Marketplace data is available in your catalog.")
    raise RuntimeError(f"CARTO table not accessible: {carto_table}") from e

# Validate required columns exist
required_columns = [
    'h3', 'population', 'urbanity',
    'male_20_to_24', 'female_20_to_24',
    'male_25_to_29', 'female_25_to_29',
    'male_30_to_34', 'female_30_to_34',
    'retail', 'food_drink', 'leisure', 'education',
    'healthcare', 'financial', 'tourism', 'transportation',
    'human_activity_index'
]

missing_columns = [col for col in required_columns if col not in carto_df.columns]

if missing_columns:
    print(f"\n❌ ERROR: Missing required columns: {missing_columns}")
    print(f"\nAvailable columns: {carto_df.columns[:20]}...")  # Show first 20
    raise ValueError(f"Missing required CARTO columns: {missing_columns}")

print(f"\n✓ All required columns present")
print(f"\nSample data:")
display(carto_df.select(required_columns[:10]).limit(5))

## Filter to Target States Using Spatial Join

In [ ]:
# Load state boundaries for target states
target_boundaries = spark.table(states_table).filter(
    col("state_abbr").isin(state_list)
)

state_count = target_boundaries.count()
if state_count == 0:
    raise ValueError(f"No states found for filter: {state_list}")

print(f"✓ Loaded {state_count} state boundaries: {state_list}")

# Generate H3 cells covering all target states at resolution 8
# Use hierarchical approach: coarse cover (res 5) → explode to res 8
target_h3_cells = target_boundaries.select(
    col("state_abbr"),
    explode(expr("h3_coverash3string(ST_AsBinary(geometry), 5)")).alias("coarse_h3")
).select(
    col("state_abbr"),
    explode(expr("h3_tochildren(coarse_h3, 8)")).alias("h3_cell_id")
).distinct()

target_h3_count = target_h3_cells.count()
print(f"✓ Generated {target_h3_count:,} H3 cells (resolution 8) covering {state_list}")

# Show distribution by state
print("\nH3 cells by state:")
display(target_h3_cells.groupBy("state_abbr").count().orderBy("state_abbr"))

In [ ]:
# Join CARTO data with target state H3 cells (spatial filter)
# Note: An H3 cell can span state boundaries, so we deduplicate by h3 after joining
carto_filtered = carto_df.join(
    target_h3_cells,
    carto_df["h3"] == target_h3_cells["h3_cell_id"],
    "inner"
).drop("h3_cell_id")  # Drop duplicate H3 column from join

# Deduplicate in case H3 cells appear in multiple states (border cells)
# Keep first state alphabetically for consistency
from pyspark.sql.window import Window
window_spec = Window.partitionBy("h3").orderBy("state_abbr")
carto_filtered = carto_filtered.withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

carto_filtered_count = carto_filtered.count()
print(f"\n✓ Filtered CARTO to target states: {carto_filtered_count:,} H3 cells")
print(f"  Coverage: {100 * carto_filtered_count / target_h3_count:.1f}% of target H3 grid")

# Show distribution by state
print("\nFiltered H3 cells by state:")
display(carto_filtered.groupBy("state_abbr").count().orderBy("state_abbr"))

## Select Relevant Columns

In [ ]:
# Define columns to include
# Only select columns needed for downstream processing
selected_columns = [
    'h3',
    'state_abbr',  # Added for multi-state support
    'geoid',
    'do_date',
    'country_iso',
    
    # Demographics - Young adults (20-34) target demographic
    'population',
    'male_20_to_24', 'female_20_to_24',
    'male_25_to_29', 'female_25_to_29',
    'male_30_to_34', 'female_30_to_34',
    
    # POI counts by category
    'retail', 'education', 'financial', 'food_drink',
    'healthcare', 'leisure', 'tourism', 'transportation',
    
    # Activity/density indicators
    'urbanity',
    'human_activity_index'
]

carto_selected = carto_filtered.select(*selected_columns)
print(f"Selected {len(selected_columns)} columns for processing")

## Create Derived Features

In [ ]:
# Derive target demographic: Young adults aged 20-34 (target market for fast, cheap pizza)
h3_features = carto_selected.withColumn(
    "target_demographic_total",
    (
        coalesce(col("male_20_to_24"), lit(0)) +
        coalesce(col("female_20_to_24"), lit(0)) +
        coalesce(col("male_25_to_29"), lit(0)) +
        coalesce(col("female_25_to_29"), lit(0)) +
        coalesce(col("male_30_to_34"), lit(0)) +
        coalesce(col("female_30_to_34"), lit(0))
    ).cast("long")
)

# Derive total POI count across all categories
h3_features = h3_features.withColumn(
    "total_poi_count",
    (
        coalesce(col("retail"), lit(0)) +
        coalesce(col("food_drink"), lit(0)) +
        coalesce(col("leisure"), lit(0)) +
        coalesce(col("education"), lit(0)) +
        coalesce(col("healthcare"), lit(0)) +
        coalesce(col("financial"), lit(0)) +
        coalesce(col("tourism"), lit(0)) +
        coalesce(col("transportation"), lit(0))
    ).cast("long")
)

print("\n✓ Created derived features:")
print("  - target_demographic_total (sum of ages 20-34)")
print("  - total_poi_count (sum of all POI categories)")

## Standardize Urbanity Categories

Map CARTO urbanity values to standardized categories for K-ring sizing:
- Very High/High density → K=2 (urban, ~0.87 mi radius)
- Medium/Low density → K=3 (suburban, ~1.45 mi radius)
- Rural/Remote → K=8 (rural, ~4.0 mi radius)

In [ ]:
# Add urbanity category for K-ring sizing
h3_features = h3_features.withColumn(
    "urbanity_category",
    when(col("urbanity").isin("Very_High_density_urban", "High_density_urban"), "urban")
    .when(col("urbanity").isin("Medium_density_urban", "Low_density_urban"), "suburban")
    .when(col("urbanity").isin("rural", "remote"), "rural")
    .otherwise("suburban")  # Default to suburban for unknowns
)

# Add K-ring size based on urbanity
h3_features = h3_features.withColumn(
    "kring_size",
    when(col("urbanity_category") == "urban", 2)
    .when(col("urbanity_category") == "suburban", 3)
    .when(col("urbanity_category") == "rural", 8)
    .otherwise(3)
)

print("\n✓ Standardized urbanity categories")
print("\nUrbanity distribution:")
display(h3_features.groupBy("urbanity", "urbanity_category", "kring_size").count().orderBy("kring_size", "urbanity"))

## Add Processing Metadata

In [ ]:
# Add processing timestamp
h3_features_final = h3_features.withColumn(
    "processing_timestamp", F.current_timestamp()
)

# Standardize H3 column name
h3_features_final = h3_features_final.withColumnRenamed("h3", "h3_cell_id")

print(f"\nFinal schema with {len(h3_features_final.columns)} columns")
print(f"Total rows: {h3_features_final.count():,}")

## Write to Silver Table

In [ ]:
# Write to Delta table
(
    h3_features_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {h3_features_final.count():,} rows to {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("H3 FEATURES VALIDATION")
print("=" * 80)

# Summary statistics
summary = spark.sql(f"""
    SELECT
        COUNT(*) as total_cells,
        COUNT(DISTINCT h3_cell_id) as unique_h3,
        COUNT(DISTINCT state_abbr) as unique_states,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(target_demographic_total), 0) as avg_target_demo,
        ROUND(AVG(total_poi_count), 0) as avg_total_poi,
        ROUND(AVG(human_activity_index), 2) as avg_activity_index
    FROM {output_table}
""")
display(summary)

# Distribution by state
print("\nH3 cells by state:")
display(spark.sql(f"""
    SELECT
        state_abbr,
        COUNT(*) as cell_count,
        ROUND(AVG(population), 0) as avg_pop,
        ROUND(AVG(target_demographic_total), 0) as avg_target_demo,
        ROUND(AVG(total_poi_count), 0) as avg_poi
    FROM {output_table}
    GROUP BY state_abbr
    ORDER BY state_abbr
"""))

# Feature distribution by urbanity
print("\nFeature distribution by urbanity category:")
urbanity_stats = spark.sql(f"""
    SELECT
        urbanity_category,
        kring_size,
        COUNT(*) as cell_count,
        ROUND(AVG(population), 0) as avg_pop,
        ROUND(AVG(target_demographic_total), 0) as avg_target_demo,
        ROUND(AVG(total_poi_count), 0) as avg_poi
    FROM {output_table}
    GROUP BY urbanity_category, kring_size
    ORDER BY kring_size
""")
display(urbanity_stats)

# Check for nulls in key columns
null_check = spark.sql(f"""
    SELECT
        COUNT(CASE WHEN h3_cell_id IS NULL THEN 1 END) as null_h3,
        COUNT(CASE WHEN population IS NULL THEN 1 END) as null_pop,
        COUNT(CASE WHEN target_demographic_total IS NULL THEN 1 END) as null_demo,
        COUNT(CASE WHEN total_poi_count IS NULL THEN 1 END) as null_poi,
        COUNT(CASE WHEN urbanity_category IS NULL THEN 1 END) as null_urbanity,
        COUNT(CASE WHEN state_abbr IS NULL THEN 1 END) as null_state
    FROM {output_table}
""")
result = null_check.collect()[0]

if any(result[col] > 0 for col in result.asDict().keys()):
    print(f"\n⚠️  WARNING: Found null values:")
    for col_name, count in result.asDict().items():
        if count > 0:
            print(f"  {col_name}: {count}")
else:
    print(f"\n✓ No null values in key columns")

# Sample data
print("\nSample cleaned H3 features:")
display(spark.table(output_table).select(
    "h3_cell_id", "state_abbr", "population", "target_demographic_total", "total_poi_count",
    "urbanity", "urbanity_category", "kring_size", "human_activity_index"
).limit(10))

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)